# Практика 12. Довірчі інтервали для середнього і частки

**Варіант 7 — Івано-Франківськ**

Липнева середньомісячна температура: **17 °C**

У роботі досліджуються довірчі інтервали для середнього значення та частки, а також вплив рівня довіри на ширину інтервалу.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

In [3]:
np.random.seed(7)

july_temperature = 17

daily_temps = np.random.normal(
    loc=july_temperature,
    scale=2.5,
    size=30
)

print("Вибірка денних температур:")
print(daily_temps)

print("\nСереднє значення:", daily_temps.mean())
print("Вибіркове стандартне відхилення:", daily_temps.std(ddof=1))

Вибірка денних температур:
[21.22631426 15.83515657 17.08205041 18.01879071 15.02769243 17.00516393
 16.99777404 12.61318923 19.54414501 18.50124629 15.43642757 16.57112935
 18.26324844 16.34660896 16.3931273  13.36689647 18.38645078 17.30970226
 17.68614981 13.18368867 21.12674923 17.38583884 16.03215014 22.07268055
 16.88653493 13.37330325 15.98693036 11.27921225 19.62349137 15.9588142 ]

Середнє значення: 16.81735525365642
Вибіркове стандартне відхилення: 2.526613018816725


### Висновок до Завдання 1

Було згенеровано вибірку з 30 денних температур для Івано-Франківська навколо липневого значення 17 °C. Для вибірки обчислено середнє значення та вибіркове стандартне відхилення з `ddof=1`.

In [4]:
mean = daily_temps.mean()
s = daily_temps.std(ddof=1)
n = len(daily_temps)

se = s / np.sqrt(n)

ci_mean = stats.t.interval(
    0.95,
    df=n - 1,
    loc=mean,
    scale=se
)

print("Середнє:", mean)
print("Вибіркове стандартне відхилення:", s)
print("Розмір вибірки:", n)
print("Стандартна похибка:", se)
print("95% довірчий інтервал для середнього:", ci_mean)

Середнє: 16.81735525365642
Вибіркове стандартне відхилення: 2.526613018816725
Розмір вибірки: 30
Стандартна похибка: 0.461294314830715
95% довірчий інтервал для середнього: (np.float64(15.873902447217345), np.float64(17.760808060095492))


### Пояснення

Для побудови довірчого інтервалу використано t-розподіл, оскільки справжнє стандартне відхилення генеральної сукупності σ невідоме. Замість нього використовується вибіркове стандартне відхилення s, обчислене за 30 спостереженнями.

Звичайне z-критичне значення тут використовувати недоцільно, оскільки умова відомого σ не виконується. t-розподіл враховує додаткову невизначеність, пов'язану з оцінюванням σ за вибіркою.

In [5]:
above_threshold = daily_temps > july_temperature

p_hat = above_threshold.mean()

n_p = n * p_hat
n_1p = n * (1 - p_hat)

se_p = np.sqrt(
    p_hat * (1 - p_hat) / n
)

ci_prop = stats.norm.interval(
    0.95,
    loc=p_hat,
    scale=se_p
)

print("Кількість днів вище 17 °C:", above_threshold.sum())
print("Оцінка частки p_hat:", p_hat)
print("n * p_hat:", n_p)
print("n * (1 - p_hat):", n_1p)
print("Стандартна похибка частки:", se_p)
print("95% довірчий інтервал для частки:", ci_prop)
print("Чи потрапляє 0.5 в інтервал:", ci_prop[0] <= 0.5 <= ci_prop[1])

Кількість днів вище 17 °C: 14
Оцінка частки p_hat: 0.4666666666666667
n * p_hat: 14.0
n * (1 - p_hat): 16.0
Стандартна похибка частки: 0.09108400680852977
95% довірчий інтервал для частки: (np.float64(0.2881452937543473), np.float64(0.6451880395789861))
Чи потрапляє 0.5 в інтервал: True


### Пояснення

Було обчислено частку днів, коли температура перевищувала липневе табличне значення 17 °C.

Для перевірки коректності нормального наближення обчислено `n * p_hat` та `n * (1 - p_hat)`. Обидва значення мають бути достатньо великими, щоб нормальне наближення було прийнятним.

Оскільки вибірка генерується симетрично навколо 17 °C, теоретична частка значень вище цього порогу становить приблизно 0.5. Якщо 0.5 потрапляє до отриманого 95% довірчого інтервалу, результат узгоджується з теоретичним значенням. Якщо не потрапляє, це може бути наслідком випадковості конкретної вибірки з 30 спостережень. При багаторазовому повторенні процедури приблизно 95% побудованих таким способом інтервалів накриватимуть справжнє значення.

In [6]:
confidence_levels = [0.90, 0.95, 0.99]

results = []

for confidence in confidence_levels:
    interval = stats.t.interval(
        confidence,
        df=n - 1,
        loc=mean,
        scale=se
    )
    
    width = interval[1] - interval[0]
    
    results.append(
        (confidence, interval[0], interval[1], width)
    )

confidence_table = pd.DataFrame(
    results,
    columns=[
        "Confidence level",
        "Lower bound",
        "Upper bound",
        "Width"
    ]
)

confidence_table

,Confidence level,Lower bound,Upper bound,Width
0,0.90,16.033558,17.601153,1.567595
1,0.95,15.873902,17.760808,1.886906
2,0.99,15.545850,18.088860,2.543010


### Пояснення

Ширина довірчого інтервалу зростає зі збільшенням рівня довіри: 90% → 95% → 99%.

Чим більший рівень довіри, тим ширший інтервал потрібен для того, щоб частіше накривати істинне значення параметра.

Щоб отримати вужчий інтервал при тому самому рівні довіри 99%, потрібно збільшити розмір вибірки n. Зі збільшенням n стандартна похибка зменшується, тому інтервал стає вужчим.

## Завдання 5. Інтерпретація 95% довірчого інтервалу

Отриманий 95% довірчий інтервал для середньої температури побудований за заданою статистичною процедурою. Якщо багато разів генерувати вибірки такого самого розміру та для кожної будувати 95% довірчий інтервал тим самим способом, приблизно 95% таких інтервалів у довгостроковій перспективі накриватимуть істинне середнє значення. Для вже отриманого конкретного інтервалу не слід говорити, що ймовірність знаходження істинного середнього в ньому дорівнює 95%.

# Контрольні питання

### 1. Яка різниця між точковою оцінкою і довірчим інтервалом?

Точкова оцінка дає одне число, яке використовується як оцінка невідомого параметра, наприклад вибіркове середнє x̄. Однак одне число не показує, наскільки точно цей параметр оцінено. Довірчий інтервал доповнює точкову оцінку діапазоном можливих значень і враховує статистичну невизначеність вибірки.

### 2. Чому для невідомого σ використовують t-розподіл?

Коли справжнє стандартне відхилення генеральної сукупності σ невідоме, воно замінюється вибірковим стандартним відхиленням s. Через це виникає додаткова невизначеність. t-розподіл враховує цю невизначеність і має важчі хвости, особливо при невеликих вибірках.

### 3. Що насправді означає 95% довіри?

95% довіри означає властивість процедури побудови інтервалу. Якщо багато разів отримувати незалежні вибірки та будувати для кожної 95% довірчий інтервал однаковим способом, приблизно 95% отриманих інтервалів накриватимуть істинне значення параметра. Не можна правильно трактувати це як 95% ймовірність того, що істинне значення знаходиться в уже побудованому конкретному інтервалі.

### 4. Які три фактори визначають ширину довірчого інтервалу?

На ширину довірчого інтервалу впливають рівень довіри, варіативність даних і розмір вибірки.

Збільшення рівня довіри робить інтервал ширшим. Збільшення стандартного відхилення також збільшує ширину інтервалу. Збільшення розміру вибірки зменшує стандартну похибку, тому інтервал стає вужчим.